In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *


In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [4]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_720s.20241120_001200_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/hmi.V_720s.20241120_001200_TAI.3.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz']

In [5]:
file_hmi = files[0]
file_fdt = files[4]

file_hmi, file_fdt

('/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz')

In [6]:
cor = np.zeros((10,10))

for i, dcrota in enumerate(np.arange(0.2,0.4,0.02)):
    for j, drsun in enumerate(np.arange(-0.,0.4,0.04)):

        with fits.open(file_hmi) as hdul:
            header_hmi = hdul[1].header
            data_hmi = hdul[1].data

        with fits.open(file_fdt) as hdul:
            header_fdt = hdul[0].header
            data_fdt = hdul[0].data

        data_fdt = undistort(data_fdt, header_fdt, xd, yd)

        data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
        data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=True, mu_thr=0.2, crota=header_fdt['CROTA'] + dcrota, rsun=330.25 + drsun, xc=421.64, yc=457.87)

        data_fdt[np.isnan(data_hmi)] = np.nan

        q = np.nanmean(data_fdt * data_hmi) / np.sqrt(np.nanmean(data_fdt ** 2) * np.nanmean(data_hmi ** 2))
        cor[i,j] = q

        print(dcrota, drsun, q)



0.2 -0.0 0.936584672434592
0.2 0.04 0.9369378479487319
0.2 0.08 0.9371351442606842
0.2 0.12 0.9372027225792449
0.2 0.16 0.9371369906274656
0.2 0.2 0.9369614620788461
0.2 0.24 0.9366421296016155
0.2 0.28 0.9361707840606261
0.2 0.32 0.9355402907210217
0.2 0.36 0.9347620641340671
0.22 -0.0 0.9399327440072099
0.22 0.04 0.9402981772522739
0.22 0.08 0.9405380175772929
0.22 0.12 0.9406469027011604
0.22 0.16 0.9405981726223411
0.22 0.2 0.9404010547915308
0.22 0.24 0.9400636102611685
0.22 0.28 0.9396063599204686
0.22 0.32 0.938993682483206
0.22 0.36 0.9382204077494725
0.24 -0.0 0.9425595543200185
0.24 0.04 0.9429715793303158
0.24 0.08 0.9432490329870091
0.24 0.12 0.9433719441474089
0.24 0.16 0.943338482470592
0.24 0.2 0.9431564234787244
0.24 0.24 0.942828333871068
0.24 0.28 0.9423576251628528
0.24 0.32 0.9417475920891067
0.24 0.36 0.9409836708210423
0.26 -0.0 0.9445038268010196
0.26 0.04 0.944949839713117
0.26 0.08 0.9452438815841128
0.26 0.12 0.9453851269407163
0.26 0.16 0.9453546348869014
0.2

In [7]:
plt.figure(figsize=(8,8))
plt.imshow(cor, cmap='gray')

In [8]:
i, j = np.where(cor == np.max(cor))
np.arange(0.2,0.4,0.02)[i], np.arange(0.,0.4,0.04)[j], cor[i,j]

(array([0.3]), array([0.12]), array([0.94722799]))